# Prueba Técnica — Ingeniería de Datos

Resolución de los ejercicios propuestos para la prueba técnica de Ingeniería de Datos.

La solución fue desarrollada utilizando PySpark, siguiendo las convenciones de nombres indicadas en el enunciado y dejando explícitos los supuestos y decisiones relevantes directamente en el código.

Los ejercicios abordan los siguientes temas:

- Transformación de datos desde Bronze a Silver.
- Identificación y corrección de problemas en un merge incremental.
- Implementación de una carga incremental utilizando watermark.

# Ejercicio 1
# Transformación de datos Bronze a Silver

Este ejercicio implementa la transformación de los datos desde la capa **Bronze** hacia la capa **Silver** utilizando PySpark.

El objetivo es tomar los datos originales conservados en Bronze y generar estructuras más limpias, normalizadas y preparadas para su posterior consumo.

Durante el proceso se realizan las siguientes tareas:

- Lectura y revisión de los datos disponibles en Bronze.
- Normalización y tipificación de los datos de clientes.
- Desanidamiento del array de compras (`purchases`) para obtener una estructura tabular.
- Tratamiento explícito de valores nulos y arrays vacíos.
- Validaciones básicas sobre los DataFrames resultantes.

Se busca preservar la información original siempre que sea posible, evitando reemplazar datos ausentes por valores ficticios.

## 1. Carga y exploración de datos Bronze

En esta etapa se cargan los datos correspondientes a la capa **Bronze** y se realiza una revisión inicial de su estructura.

El objetivo es trabajar sobre los datos originales antes de aplicar reglas de transformación o normalización.

Se revisan principalmente:

- Esquema y tipos de datos recibidos.
- Columnas disponibles.
- Estructuras anidadas (`struct`).
- Arrays como `purchases` y `contact.phones`.
- Presencia de valores nulos.
- Cantidad de registros disponibles.

En esta capa no se modifica el significado de los datos; Bronze conserva la información recibida desde el origen.

In [0]:
import pyspark.sql.functions as F

bronze_path = (
    "file:/Workspace/Users/silva.nicolasn@gmail.com/"
    "Drafts/customers_bronze.json"
)

customers_bronze_df = (
    spark.read
    .option("multiline", True)
    .json(bronze_path)
)

display(customers_bronze_df)

customers_bronze_df.printSchema()

_id,contact,country,created_at,full_name,is_active,purchases
60c1f2a1e4b0a1a2b3c4d5e6,"List(maria.lopez@example.com, List(+50255512345, +50255598765))",GT,2024-03-11T14:22:00Z,María López,true,"List(List(129.5, GTQ, ORD-001), List(45.0, GTQ, ORD-002))"
60c1f2a1e4b0a1a2b3c4d5e7,"List(null, List())",SV,null,Carlos Reyes,1,List()
60c1f2a1e4b0a1a2b3c4d5e8,"List(ana.torres@example.com, List(+593987654321))",ec,2024-03-12 09:15:00,Ana Torres,false,"List(List(88.25, USD, ORD-003))"
60c1f2a1e4b0a1a2b3c4d5e9,"List(sin_nombre@example.com, List(+51987654321))",PE,2024-03-13T08:00:00Z,null,true,"List(List(200.0, PEN, ORD-004), List(null, PEN, ORD-005))"


root
 |-- _id: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phones: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- country: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- purchases: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- currency: string (nullable = true)
 |    |    |-- order_id: string (nullable = true)



## 2. Transformación y normalización de clientes

A partir del DataFrame Bronze se construye el DataFrame de clientes correspondiente a la capa **Silver**.

En esta etapa se normalizan los principales atributos del cliente para obtener una estructura consistente y preparada para análisis posteriores.

Las principales transformaciones son:

- `_id` se conserva como identificador del cliente (`customer_id`).
- Los textos se limpian utilizando `trim`.
- Los códigos y campos categóricos se estandarizan utilizando mayúsculas cuando corresponde.
- Las fechas se convierten a tipos de datos adecuados.
- Los datos anidados de contacto y dirección se extraen a columnas independientes.
- El estado `is_active` se normaliza a un valor booleano.
- Los valores nulos se conservan cuando no existe información válida en el origen.

El resultado es un DataFrame Silver con una estructura más limpia, tipificada y consistente.

In [0]:
customers_silver_df = (
    customers_bronze_df
    .select(

        # Renombramos el identificador del cliente.
        F.col("_id").alias("customer_id"),

        # Estandarizamos país: sin espacios y en mayúsculas.
        F.upper(
            F.trim("country")
        ).alias("pais_cd"),

        # Limpiamos espacios del nombre.
        # NULL se conserva.
        F.trim("full_name")
        .alias("nombre_txt"),

        # Extraemos y limpiamos el email.
        # NULL se conserva.
        F.trim("contact.email")
        .alias("email_txt"),

        # Limpiamos cada teléfono manteniendo el array.
        # [] y NULL se conservan.
        F.transform(
            F.col("contact.phones"),
            lambda telefono: F.trim(telefono)
        ).alias("telefonos_arr"),

        # Unificamos los formatos de fecha a timestamp.
        # Si no es válido, queda NULL.
        F.coalesce(
            F.try_to_timestamp(
                "created_at",
                F.lit("yyyy-MM-dd'T'HH:mm:ssX")
            ),
            F.try_to_timestamp(
                "created_at",
                F.lit("yyyy-MM-dd HH:mm:ss")
            )
        ).alias("creado_ts"),

        # Normalizamos is_active a boolean.
        # true/1 -> True, false/0 -> False.
        # Otros valores quedan NULL.
        F.when(
            F.lower(
                F.trim(
                    F.col("is_active").cast("string")
                )
            ).isin("true", "1"),
            True
        )
        .when(
            F.lower(
                F.trim(
                    F.col("is_active").cast("string")
                )
            ).isin("false", "0"),
            False
        )
        .otherwise(
            F.lit(None).cast("boolean")
        )
        .alias("activo_flag")
    )
)

In [0]:
display(customers_silver_df)

customers_silver_df.printSchema()

customer_id,pais_cd,nombre_txt,email_txt,telefonos_arr,creado_ts,activo_flag
60c1f2a1e4b0a1a2b3c4d5e6,GT,María López,maria.lopez@example.com,"List(+50255512345, +50255598765)",2024-03-11T14:22:00.000Z,true
60c1f2a1e4b0a1a2b3c4d5e7,SV,Carlos Reyes,null,List(),null,true
60c1f2a1e4b0a1a2b3c4d5e8,EC,Ana Torres,ana.torres@example.com,List(+593987654321),2024-03-12T09:15:00.000Z,false
60c1f2a1e4b0a1a2b3c4d5e9,PE,null,sin_nombre@example.com,List(+51987654321),2024-03-13T08:00:00.000Z,true


root
 |-- customer_id: string (nullable = true)
 |-- pais_cd: string (nullable = true)
 |-- nombre_txt: string (nullable = true)
 |-- email_txt: string (nullable = true)
 |-- telefonos_arr: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- creado_ts: timestamp (nullable = true)
 |-- activo_flag: boolean (nullable = true)



## 3. Normalización del array de compras

Los datos Bronze contienen las compras de cada cliente dentro del array `purchases`.

Para facilitar su análisis y mantener un modelo tabular en Silver, cada elemento del array se transforma en una fila independiente utilizando `explode`.

Durante esta transformación:

- Se conserva `customer_id` para mantener la relación con el cliente.
- Se conserva el código de país normalizado (`pais_cd`).
- Cada elemento de `purchases` genera una fila independiente.
- `order_id` se limpia eliminando espacios innecesarios.
- `amount` se conserva con su valor original, incluyendo `null`.
- `currency` se normaliza eliminando espacios y utilizando mayúsculas.

De esta forma se obtiene un DataFrame Silver con una granularidad de **una fila por compra de cada cliente**.

In [0]:
customer_purchases_silver_df = (
    customers_bronze_df

    .select(
        F.col("_id").alias("customer_id"),

        F.upper(
            F.trim("country")
        ).alias("pais_cd"),

        F.explode("purchases")
        .alias("purchase")
    )

    .select(
        "customer_id",
        "pais_cd",

        F.trim("purchase.order_id")
        .alias("order_id_txt"),

        F.col("purchase.amount")
        .alias("monto_val"),

        F.upper(
            F.trim("purchase.currency")
        ).alias("moneda_cd")
    )
)

In [0]:
display(customer_purchases_silver_df)

customer_purchases_silver_df.printSchema()

customer_id,pais_cd,order_id_txt,monto_val,moneda_cd
60c1f2a1e4b0a1a2b3c4d5e6,GT,ORD-001,129.5,GTQ
60c1f2a1e4b0a1a2b3c4d5e6,GT,ORD-002,45.0,GTQ
60c1f2a1e4b0a1a2b3c4d5e8,EC,ORD-003,88.25,USD
60c1f2a1e4b0a1a2b3c4d5e9,PE,ORD-004,200.0,PEN
60c1f2a1e4b0a1a2b3c4d5e9,PE,ORD-005,null,PEN


root
 |-- customer_id: string (nullable = true)
 |-- pais_cd: string (nullable = true)
 |-- order_id_txt: string (nullable = true)
 |-- monto_val: double (nullable = true)
 |-- moneda_cd: string (nullable = true)




## 4. Tratamiento de valores nulos y arrays vacíos

Durante la transformación de Bronze a Silver se aplican las siguientes reglas:

- `full_name = null` → `nombre_txt = null`.
- `contact.email = null` → `email_txt = null`.
- `created_at = null` o con formato no reconocido → `creado_ts = null`.
- `purchase.amount = null` → `monto_val = null`; un monto desconocido no se interpreta como cero.
- `contact.phones = []` → se conserva como array vacío; si viene `null`, se mantiene como `null`.
- `purchases = []` o `null` → no genera registros en la tabla de compras Silver.
- `is_active`: `true/1` → `True`, `false/0` → `False` y valores no reconocidos → `null`.

Como criterio general, los valores ausentes se conservan sin reemplazarlos por información ficticia.

### Validación de valores nulos

Se valida que los valores ausentes de nombre, email y fecha continúen como `null` después de la transformación a Silver.

In [0]:
# Validamos que los valores NULL del origen
# se conserven como NULL en Silver.
display(
    customers_bronze_df.alias("b")
    .join(
        customers_silver_df.alias("s"),
        f.col("b._id") == f.col("s.customer_id"),
        "inner"
    )
    .filter(
        f.col("b.full_name").isNull()
        | f.col("b.contact.email").isNull()
        | f.col("b.created_at").isNull()
    )
    .select(
        f.col("s.customer_id"),
        f.col("b.full_name").alias("full_name_origen"),
        f.col("s.nombre_txt"),
        f.col("b.contact.email").alias("email_origen"),
        f.col("s.email_txt"),
        f.col("b.created_at").alias("created_at_origen"),
        f.col("s.creado_ts")
    )
)

customer_id,full_name_origen,nombre_txt,email_origen,email_txt,created_at_origen,creado_ts
60c1f2a1e4b0a1a2b3c4d5e7,Carlos Reyes,Carlos Reyes,null,null,null,null
60c1f2a1e4b0a1a2b3c4d5e9,null,null,sin_nombre@example.com,sin_nombre@example.com,2024-03-13T08:00:00Z,2024-03-13T08:00:00.000Z


### Validación de teléfonos

Se comprueba que un array vacío permanezca como `[]` y que un valor `null` continúe siendo `null`.

In [0]:
# Validamos arrays vacíos y NULL en teléfonos.
display(
    customers_silver_df
    .filter(
        f.col("telefonos_arr").isNull()
        | (f.size("telefonos_arr") == 0)
    )
    .select(
        "customer_id",
        "pais_cd",
        "telefonos_arr"
    )
)

customer_id,pais_cd,telefonos_arr
60c1f2a1e4b0a1a2b3c4d5e7,SV,List()


### Validación de montos

Se valida que los montos ausentes se conserven como `null`, evitando interpretar un valor desconocido como cero.

In [0]:
# Validamos que amount NULL
# permanezca como monto_val NULL.
display(
    customer_purchases_silver_df
    .filter(
        f.col("monto_val").isNull()
    )
    .select(
        "customer_id",
        "pais_cd",
        "order_id_txt",
        "monto_val",
        "moneda_cd"
    )
)

customer_id,pais_cd,order_id_txt,monto_val,moneda_cd
60c1f2a1e4b0a1a2b3c4d5e9,PE,ORD-005,null,PEN


### Validación de is_active

Se valida la normalización de los valores disponibles de `is_active`. Las representaciones `true/1` se convierten a `True` y `false/0` a `False`. Cualquier valor no reconocido se define como `null`.

In [0]:
# Validamos la normalización de is_active:
# true/1  -> True
# false/0 -> False
# otros   -> NULL
display(
    customers_bronze_df.alias("b")
    .join(
        customers_silver_df.alias("s"),
        f.col("b._id") == f.col("s.customer_id"),
        "inner"
    )
    .select(
        f.col("s.customer_id"),
        f.col("b.is_active").alias("is_active_origen"),
        f.col("s.activo_flag")
    )
)

customer_id,is_active_origen,activo_flag
60c1f2a1e4b0a1a2b3c4d5e6,true,true
60c1f2a1e4b0a1a2b3c4d5e7,1,true
60c1f2a1e4b0a1a2b3c4d5e8,false,false
60c1f2a1e4b0a1a2b3c4d5e9,true,true


### Validación de arrays de compras vacíos

Los clientes cuyo array `purchases` está vacío o es `null` no generan registros en la tabla de compras Silver, ya que su granularidad es una fila por compra existente.

In [0]:
# Identificamos clientes sin compras en Bronze.
customers_without_purchases_df = (
    customers_bronze_df
    .filter(
        f.col("purchases").isNull()
        | (f.size("purchases") == 0)
    )
    .select(
        f.col("_id").alias("customer_id"),
        "country",
        "purchases"
    )
)

display(customers_without_purchases_df)

customer_id,country,purchases
60c1f2a1e4b0a1a2b3c4d5e7,SV,List()


### Validación de compras vacías

Se identifican clientes cuyo array `purchases` está vacío o es `null` y se valida que no hayan generado registros en `customer_purchases_silver_df`, ya que cada fila Silver representa una compra existente.

In [0]:
# Validamos que clientes sin compras
# no hayan generado registros en Silver.
display(
    customers_without_purchases_df.alias("b")
    .join(
        customer_purchases_silver_df.alias("s"),
        f.col("b.customer_id") == f.col("s.customer_id"),
        "inner"
    )
    .select(
        f.col("b.customer_id"),
        f.col("b.country"),
        f.col("b.purchases"),
        f.col("s.order_id_txt")
    )
)

customer_id,country,purchases,order_id_txt


## 5. Particionamiento por país y fecha

El enunciado solicita particionar el resultado por país y fecha, pero no especifica qué fecha debe utilizarse.

Para esta solución se utiliza `ingesta_dt`, que representa la fecha técnica en la que los datos son procesados desde Bronze hacia Silver.

`ingesta_dt` no representa:

- la fecha de creación del cliente (`creado_ts`);
- una fecha de compra.

Se utiliza exclusivamente como metadata técnica de la carga y como columna de particionamiento.

Las tablas Silver se particionan por:

`pais_cd + ingesta_dt`

In [0]:
# Obtenemos una única fecha técnica para toda la ejecución
# Bronze -> Silver.
ingesta_dt = (
    spark.sql(
        "SELECT current_date() AS ingesta_dt"
    )
    .first()["ingesta_dt"]
)

print("Fecha de ingesta:", ingesta_dt)

Fecha de ingesta: 2026-08-13


In [0]:
customers_silver_final_df = (
    customers_silver_df

    # Metadata técnica de la carga.
    # Se utiliza para particionamiento y no representa
    # una fecha de negocio del cliente.
    .withColumn(
        "ingesta_dt",
        f.lit(ingesta_dt).cast("date")
    )
)

display(customers_silver_final_df)

customer_id,pais_cd,nombre_txt,email_txt,telefonos_arr,creado_ts,activo_flag,ingesta_dt
60c1f2a1e4b0a1a2b3c4d5e6,GT,María López,maria.lopez@example.com,"List(+50255512345, +50255598765)",2024-03-11T14:22:00.000Z,true,2026-08-13
60c1f2a1e4b0a1a2b3c4d5e7,SV,Carlos Reyes,null,List(),null,true,2026-08-13
60c1f2a1e4b0a1a2b3c4d5e8,EC,Ana Torres,ana.torres@example.com,List(+593987654321),2024-03-12T09:15:00.000Z,false,2026-08-13
60c1f2a1e4b0a1a2b3c4d5e9,PE,null,sin_nombre@example.com,List(+51987654321),2024-03-13T08:00:00.000Z,true,2026-08-13


In [0]:
customer_purchases_silver_final_df = (
    customer_purchases_silver_df

    # Misma fecha técnica utilizada para customers_silver.
    .withColumn(
        "ingesta_dt",
        f.lit(ingesta_dt).cast("date")
    )
)

display(customer_purchases_silver_final_df)

customer_id,pais_cd,order_id_txt,monto_val,moneda_cd,ingesta_dt
60c1f2a1e4b0a1a2b3c4d5e6,GT,ORD-001,129.5,GTQ,2026-08-13
60c1f2a1e4b0a1a2b3c4d5e6,GT,ORD-002,45.0,GTQ,2026-08-13
60c1f2a1e4b0a1a2b3c4d5e8,EC,ORD-003,88.25,USD,2026-08-13
60c1f2a1e4b0a1a2b3c4d5e9,PE,ORD-004,200.0,PEN,2026-08-13
60c1f2a1e4b0a1a2b3c4d5e9,PE,ORD-005,null,PEN,2026-08-13


In [0]:
(
    customers_silver_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy(
        "pais_cd",
        "ingesta_dt"
    )
    .saveAsTable(
        "customers_silver"
    )
)

In [0]:
(
    customer_purchases_silver_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy(
        "pais_cd",
        "ingesta_dt"
    )
    .saveAsTable(
        "customer_purchases_silver"
    )
)

In [0]:
display(
    spark.sql(
        "DESCRIBE DETAIL customers_silver"
    )
)

display(
    spark.sql(
        "DESCRIBE DETAIL customer_purchases_silver"
    )
)

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,1dbc375f-6b10-42f8-9ccc-2c3408a45cb3,workspace.default.customers_silver,null,,2026-08-13T23:45:04.973Z,2026-08-13T23:45:14.000Z,"List(pais_cd, ingesta_dt)",List(),4,10511,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,3c3a2b57-aef2-415f-bea9-44f43b65ccd4,workspace.default.customer_purchases_silver,null,,2026-08-13T23:45:34.065Z,2026-08-13T23:45:36.000Z,"List(pais_cd, ingesta_dt)",List(),3,6192,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


### Validación del particionamiento

Se verificó mediante `DESCRIBE DETAIL` que ambas tablas Delta quedaron particionadas por:

- `pais_cd`
- `ingesta_dt`

`ingesta_dt` representa la fecha técnica de ingesta de Bronze a Silver.

# Ejercicio 2
# Corrección del merge incremental de orders_silver

Este ejercicio revisa la función `merge_orders` utilizada para realizar la carga incremental de `orders_silver`.

El objetivo es identificar las causas que pueden generar duplicados, cruces entre países y lecturas innecesarias, y luego corregir la función manteniendo la lógica general del proceso.

Durante la revisión se consideran los siguientes puntos:

- Clave utilizada para identificar una orden durante el `MERGE`.
- Cardinalidad del join contra `customers_silver`.
- Estrategia de particionamiento del backup.
- Condición utilizada para actualizar registros existentes.
- Supuestos necesarios para evitar sobrescribir información más reciente con datos antiguos.

## 6. Clave utilizada en el MERGE

La condición original utiliza únicamente `order_id_txt`:

```text
t.order_id_txt = s.order_id_txt
```

El problema es que `order_id_txt` no es único de forma global. Cada país genera sus identificadores de manera independiente, por lo que una orden de Guatemala y una orden de Honduras pueden compartir el mismo valor.

Si una fila del target ya existe para `ORD-001 / GT` y luego llega `ORD-001 / HN`, el `MERGE` puede considerar ambas filas como la misma orden. Con `whenMatchedUpdateAll()` se podrían reemplazar los datos de Guatemala por los de Honduras.

Además, si varias filas del source con el mismo `order_id_txt` coinciden con una misma fila existente del target, Delta Lake puede rechazar el `MERGE` por existir más de una fila source intentando modificar la misma fila target.

Por este motivo, la orden debe identificarse mediante una clave compuesta:

- `order_id_txt`
- `pais_cd`

La condición correcta queda conceptualmente como:

```text
t.order_id_txt = s.order_id_txt
AND t.pais_cd = s.pais_cd
```

De esta forma, un mismo identificador puede existir en países distintos sin que ambos registros se mezclen.

## 7. Cardinalidad del join contra customers_silver

`customers_silver` todavía contiene registros históricos duplicados. Si una orden se une directamente contra una tabla que contiene más de una fila para el mismo cliente, el join genera una fila por cada coincidencia encontrada.

Por ejemplo, si una orden tiene `customer_id = C-10` y `customers_silver` contiene tres filas para ese mismo cliente, el resultado del join puede contener tres copias de la orden.

La corrección se aplica antes del join, reduciendo `customers_silver` a una fila por:

`customer_id + pais_cd`

También se incorpora `pais_cd` a la condición del join. Esto evita que un cliente con el mismo identificador en dos países distintos se utilice para enriquecer una orden del país incorrecto.

Para el join se utiliza:

```python
on=["customer_id", "pais_cd"]
```

Cuando las columnas utilizadas como clave tienen el mismo nombre en ambos DataFrames, esta forma permite conservar una sola copia de las columnas de join en el resultado.

Como el enunciado no define una columna de versión para decidir cuál de los duplicados históricos es el registro vigente, se asume que para este ejercicio basta con conservar una fila por `customer_id + pais_cd`. En un proceso productivo, si los duplicados pueden contener valores distintos, se debería definir una regla determinística utilizando una fecha de actualización o una versión del registro.

## 8. Particionamiento del backup

El backup original se particiona únicamente por `updated_dt`.

Con esta estructura, los registros de los 9 países quedan mezclados dentro de las mismas particiones de fecha. Si una consulta o proceso filtra por `pais_cd`, Spark no puede descartar particiones utilizando esa columna y debe revisar datos de todos los países contenidos en las fechas seleccionadas.

Para que el particionamiento represente mejor la forma en que el equipo opera los datos, el backup se particiona por:

`pais_cd + updated_dt`

De esta forma, una consulta que filtre por país puede aprovechar `partition pruning` y evitar leer particiones correspondientes a otros países.

Además, `new_batch_df` contiene `updated_ts`, pero no `updated_dt`. Por lo tanto, antes de escribir el backup se deriva la fecha mediante `to_date(updated_ts)`.

Se conserva el modo `append`, ya que el código original trata el backup como una acumulación de los batches procesados. Si se necesitara que esa escritura también fuera idempotente frente a reintentos, sería necesario definir una regla adicional de deduplicación o un identificador de batch.

## 9. Uso de whenMatchedUpdateAll()

`whenMatchedUpdateAll()` puede utilizarse sin una condición adicional cuando se garantiza que la fila recibida representa el estado que debe quedar en Silver.

Para que sea seguro en este caso, se necesitaría asegurar principalmente que:

- Cada clave de negocio tenga como máximo una fila relevante en el source.
- Los eventos lleguen en orden o que el source siempre entregue la versión vigente.
- No pueda llegar posteriormente un registro más antiguo para una orden ya actualizada.

Reprocesar exactamente el mismo registro no es necesariamente un problema, ya que volvería a escribir el mismo estado. El riesgo aparece cuando un reproceso, evento tardío o batch antiguo contiene un `updated_ts` menor al que ya existe en el target.

Por este motivo, en la función corregida se agrega una condición temporal:

```text
s.updated_ts IS NOT NULL
AND (
    t.updated_ts IS NULL
    OR s.updated_ts > t.updated_ts
)
```

Así, una orden existente solo se actualiza cuando el registro recibido contiene una versión temporalmente más reciente.

## Función corregida

La función mantiene el flujo original, pero aplica las correcciones antes de ejecutar el `MERGE` y antes de escribir el backup.

Los principales cambios son:

- `customers_silver` se reduce a una fila por `customer_id + pais_cd` antes del join.
- El enriquecimiento se realiza utilizando `customer_id + pais_cd`.
- El `MERGE` utiliza `order_id_txt + pais_cd` como clave de negocio.
- Los registros existentes solo se actualizan cuando `updated_ts` es más reciente.
- `updated_dt` se deriva desde `updated_ts`.
- El backup se particiona por `pais_cd + updated_dt`.

Se asume que el batch de órdenes contiene como máximo una versión por `order_id_txt + pais_cd`. Si el origen pudiera enviar varias versiones de una misma orden dentro del mismo batch, sería necesario deduplicarlo por `updated_ts` antes del `MERGE`.

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable


def merge_orders(
    spark,
    new_batch_df,
    target_table_path
):
    """
    Hace merge incremental de nuevas órdenes a la tabla Silver.
    new_batch_df: order_id_txt, customer_id, pais_cd, monto_dec, updated_ts
    """

    # Se asume que los duplicados históricos de customers_silver
    # pueden reducirse a una fila por cliente y país para este ejercicio.
    customers_df = (
        spark.table("customers_silver")
        .select(
            "customer_id",
            "pais_cd",
            "nombre_txt",
            "email_txt"
        )
        .dropDuplicates(
            [
                "customer_id",
                "pais_cd"
            ]
        )
    )


    # Unimos por cliente y país para evitar multiplicar órdenes
    # con registros correspondientes a otro país.
    enriched_df = (
        new_batch_df
        .join(
            customers_df,
            on=[
                "customer_id",
                "pais_cd"
            ],
            how="left"
        )
    )


    # La orden se identifica por order_id_txt + pais_cd,
    # ya que order_id_txt puede repetirse entre países.
    merge_keys = [
        "order_id_txt",
        "pais_cd"
    ]

    merge_condition = " AND ".join(
        [
            f"t.{key} = s.{key}"
            for key in merge_keys
        ]
    )


    target = DeltaTable.forPath(
        spark,
        target_table_path
    )


    (
        target.alias("t")
        .merge(
            enriched_df.alias("s"),
            merge_condition
        )

        # Solo actualizamos cuando el registro recibido
        # contiene una versión más reciente.
        .whenMatchedUpdateAll(
            condition="""
                s.updated_ts IS NOT NULL
                AND (
                    t.updated_ts IS NULL
                    OR s.updated_ts > t.updated_ts
                )
            """
        )

        # Si la clave no existe en Silver,
        # insertamos la nueva orden.
        .whenNotMatchedInsertAll()
        .execute()
    )


    # updated_dt se deriva desde updated_ts
    # para utilizarla como fecha de particionamiento.
    backup_df = (
        enriched_df
        .withColumn(
            "updated_dt",
            F.to_date("updated_ts")
        )
    )


    # El backup se organiza por país y fecha
    # para facilitar el pruning por pais_cd.
    (
        backup_df.write
        .format("delta")
        .mode("append")
        .partitionBy(
            "pais_cd",
            "updated_dt"
        )
        .save(
            target_table_path + "_backup"
        )
    )


    return enriched_df

### Resultado de la corrección

Con estos cambios:

- Órdenes con el mismo `order_id_txt` pueden coexistir en países distintos.
- El histórico duplicado de clientes deja de multiplicar las filas antes del `MERGE`.
- El enriquecimiento se realiza utilizando el cliente del país correspondiente.
- Un registro antiguo no reemplaza una versión más reciente de la orden.
- El backup queda organizado por país y fecha, permitiendo descartar particiones de otros países cuando la consulta filtra por `pais_cd`.

La solución mantiene los supuestos explícitos dentro del notebook y concentra las reglas de negocio cerca de la transformación que las utiliza.

# Ejercicio 3 — Carga incremental con watermark

Se implementa una carga incremental sobre `inventory_silver` utilizando `updated_ts` como watermark.

El proceso debe:

- Crear la tabla si aún no existe.
- Identificar cada registro mediante `producto_id + pais_cd`.
- Insertar productos nuevos.
- Actualizar productos existentes solo si el registro recibido es más reciente.
- Evitar que datos antiguos sobrescriban información actual.

## Preparación de datos de prueba

Se crean los datos entregados en el enunciado para simular el estado actual de `inventory_silver` y el nuevo batch recibido.

La tabla histórica se almacenará en formato Delta para representar el estado actual de Silver, mientras que el batch se mantendrá como DataFrame para utilizarlo como entrada de la carga incremental.

In [0]:
from pyspark.sql import functions as F

# Datos históricos que representan el estado actual
# de inventory_silver antes de recibir el nuevo batch.
inventory_historico_data = [
    ("P-100", "GT", 45, "2024-05-01 08:00:00"),
    ("P-101", "GT", 12, "2024-05-01 08:00:00"),
    ("P-100", "HN", 20, "2024-05-01 08:00:00"),
    ("P-102", "AR", 8,  "2024-04-30 17:30:00")
]

# Creamos el DataFrame histórico.
inventory_historico_df = (
    spark.createDataFrame(
        inventory_historico_data,
        [
            "producto_id",
            "pais_cd",
            "stock_int",
            "updated_ts"
        ]
    )

    # Convertimos updated_ts de string a timestamp
    # para poder comparar fechas correctamente.
    .withColumn(
        "updated_ts",
        F.to_timestamp("updated_ts")
    )
)

display(inventory_historico_df)

producto_id,pais_cd,stock_int,updated_ts
P-100,GT,45,2024-05-01T08:00:00.000Z
P-101,GT,12,2024-05-01T08:00:00.000Z
P-100,HN,20,2024-05-01T08:00:00.000Z
P-102,AR,8,2024-04-30T17:30:00.000Z


### Creación del batch recibido

Se crea el batch del día con los nuevos registros que serán procesados por la carga incremental.

El batch incluye un registro antiguo para `P-100/HN`, utilizado para comprobar que el watermark evita sobrescribir información más reciente.

In [0]:
# Datos correspondientes al nuevo batch recibido.
inventory_batch_data = [
    ("P-100", "GT", 40, "2024-05-02 09:10:00"),
    ("P-103", "GT", 5,  "2024-05-02 09:12:00"),
    ("P-100", "HN", 20, "2024-04-29 10:00:00"),
    ("P-102", "AR", 3,  "2024-05-02 10:00:00")
]

# Creamos el DataFrame que simula el batch Bronze recibido.
inventory_bronze_batch_df = (
    spark.createDataFrame(
        inventory_batch_data,
        [
            "producto_id",
            "pais_cd",
            "stock_int",
            "updated_ts"
        ]
    )

    # Convertimos updated_ts a timestamp.
    .withColumn(
        "updated_ts",
        F.to_timestamp("updated_ts")
    )
)

display(inventory_bronze_batch_df)

producto_id,pais_cd,stock_int,updated_ts
P-100,GT,40,2024-05-02T09:10:00.000Z
P-103,GT,5,2024-05-02T09:12:00.000Z
P-100,HN,20,2024-04-29T10:00:00.000Z
P-102,AR,3,2024-05-02T10:00:00.000Z


### Creación de la tabla Delta inicial

Se crea un Volume para almacenar los datos del ejercicio.

Dentro del Volume se define una ruta para `inventory_silver` y se escribe el histórico en formato Delta, representando el estado inicial de la tabla antes de ejecutar la carga incremental.

In [0]:
# Creamos un Volume para almacenar los archivos del ejercicio.
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.default.ejercicio3
""")


# Ruta donde se almacenará inventory_silver.
target_table_path = (
    "/Volumes/workspace/default/ejercicio3/inventory_silver"
)


# Guardamos el histórico en formato Delta.
# En este momento se crea la tabla Delta en la ruta indicada.
(
    inventory_historico_df.write
    .format("delta")
    .mode("overwrite")
    .save(target_table_path)
)


# Validamos que inventory_silver contenga
# los 4 registros históricos iniciales.
display(
    spark.read
    .format("delta")
    .load(target_table_path)
)

producto_id,pais_cd,stock_int,updated_ts
P-100,GT,45,2024-05-01T08:00:00.000Z
P-101,GT,12,2024-05-01T08:00:00.000Z
P-100,HN,20,2024-05-01T08:00:00.000Z
P-102,AR,8,2024-04-30T17:30:00.000Z


## 10. Configuración de la carga incremental

Se define la clave de negocio utilizada para identificar cada registro y la columna que controlará el watermark.

- Clave de negocio: `producto_id + pais_cd`
- Watermark: `updated_ts`

In [0]:
# Configuración utilizada por la carga incremental.
conf = {
    "merge_keys": [
        "producto_id",
        "pais_cd"
    ],
    "watermark_col": "updated_ts"
}

## 11. Carga incremental

La función valida si la tabla Delta ya existe.

- Si no existe, la crea con el batch recibido.
- Si existe, realiza un `MERGE` utilizando la clave de negocio.
- Los registros nuevos se insertan.
- Los registros existentes solo se actualizan cuando el `updated_ts` recibido es más reciente.

De esta forma se evita que datos antiguos sobrescriban información vigente.

In [0]:
from delta.tables import DeltaTable


def incremental_load(
    spark,
    batch_df,
    target_table_path,
    conf
):

    # Obtenemos la configuración de la carga.
    merge_keys = conf["merge_keys"]
    watermark_col = conf["watermark_col"]


    # Construimos la condición del MERGE
    # utilizando todas las claves de negocio.
    merge_condition = " AND ".join(
        [
            f"t.{key} = s.{key}"
            for key in merge_keys
        ]
    )


    # Si la tabla Delta no existe,
    # se crea utilizando el primer batch.
    if not DeltaTable.isDeltaTable(
        spark,
        target_table_path
    ):
        (
            batch_df.write
            .format("delta")
            .mode("overwrite")
            .save(target_table_path)
        )

        return


    # Referenciamos la tabla Delta existente.
    target = DeltaTable.forPath(
        spark,
        target_table_path
    )


    # Ejecutamos la carga incremental.
    (
        target.alias("t")
        .merge(
            batch_df.alias("s"),
            merge_condition
        )

        # Si el registro existe, solo se actualiza
        # cuando el dato recibido es más reciente.
        .whenMatchedUpdateAll(
            condition=f"""
                t.{watermark_col} IS NULL
                OR s.{watermark_col} > t.{watermark_col}
            """
        )

        # Si la clave no existe, insertamos el registro.
        .whenNotMatchedInsertAll()

        .execute()
    )

In [0]:
incremental_load(
    spark,
    inventory_bronze_batch_df,
    target_table_path,
    conf
)

In [0]:
# Leemos inventory_silver después del MERGE.
inventory_result_df = (
    spark.read
    .format("delta")
    .load(target_table_path)
)

display(
    inventory_result_df
    .orderBy(
        "pais_cd",
        "producto_id"
    )
)

producto_id,pais_cd,stock_int,updated_ts
P-102,AR,3,2024-05-02T10:00:00.000Z
P-100,GT,40,2024-05-02T09:10:00.000Z
P-101,GT,12,2024-05-01T08:00:00.000Z
P-103,GT,5,2024-05-02T09:12:00.000Z
P-100,HN,20,2024-05-01T08:00:00.000Z


In [0]:
# Revisamos los registros afectados por el batch.
display(
    inventory_result_df
    .filter(
        (
            (F.col("producto_id") == "P-100")
            & F.col("pais_cd").isin("GT", "HN")
        )
        |
        (
            (F.col("producto_id") == "P-102")
            & (F.col("pais_cd") == "AR")
        )
        |
        (
            (F.col("producto_id") == "P-103")
            & (F.col("pais_cd") == "GT")
        )
    )
    .orderBy(
        "pais_cd",
        "producto_id"
    )
)

producto_id,pais_cd,stock_int,updated_ts
P-102,AR,3,2024-05-02T10:00:00.000Z
P-100,GT,40,2024-05-02T09:10:00.000Z
P-103,GT,5,2024-05-02T09:12:00.000Z
P-100,HN,20,2024-05-01T08:00:00.000Z


## 12. Resultado final de inventory_silver

Después de ejecutar la carga incremental:

- `P-100 / GT` se actualiza de stock `45` a `40`, ya que el registro recibido tiene un `updated_ts` más reciente.
- `P-103 / GT` se inserta porque no existía previamente en `inventory_silver`.
- `P-100 / HN` mantiene stock `20` y fecha `2024-05-01 08:00:00`, ya que el registro recibido es más antiguo.
- `P-102 / AR` se actualiza de stock `8` a `3`, porque el nuevo registro tiene un `updated_ts` más reciente.
- `P-101 / GT` no recibe cambios y conserva su información histórica.

El resultado final contiene 5 registros.

In [0]:
# Leemos el estado final de inventory_silver.
inventory_result_df = (
    spark.read
    .format("delta")
    .load(target_table_path)
)

# Mostramos el resultado ordenado por país y producto.
display(
    inventory_result_df
    .orderBy(
        "pais_cd",
        "producto_id"
    )
)

producto_id,pais_cd,stock_int,updated_ts
P-102,AR,3,2024-05-02T10:00:00.000Z
P-100,GT,40,2024-05-02T09:10:00.000Z
P-101,GT,12,2024-05-01T08:00:00.000Z
P-103,GT,5,2024-05-02T09:12:00.000Z
P-100,HN,20,2024-05-01T08:00:00.000Z
